## Coleta de Dados

### Sobre os Dados

The dataset contains transactions made by credit cards in September 2013 by European cardholders.
This dataset presents transactions that occurred in two days, where we have 492 frauds out of 284,807 transactions. The dataset is highly unbalanced, the positive class (frauds) account for 0.172% of all transactions.

ref: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download

Feature 'Time' contains the seconds elapsed between each transaction and the first transaction in the dataset. The feature 'Amount' is the transaction Amount, this feature can be used for example-dependant cost-sensitive learning. Feature 'Class' is the response variable and it takes value 1 in case of fraud and 0 otherwise.

Given the class imbalance ratio, we recommend measuring the accuracy using the Area Under the Precision-Recall Curve (AUPRC). Confusion matrix accuracy is not meaningful for unbalanced classification.

ref: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud?resource=download

In [ ]:
import pandas as pd
import sys

from pathlib import Path
import subprocess

script_path = Path("../scripts/download_csv.py")
data_path = Path("../../dados/creditcard.csv")

if not data_path.exists():
    print("Baixando dataset...")
    subprocess.run([sys.executable, str(script_path)], check=True)

if not data_path.exists():
    raise FileNotFoundError(
        f"Arquivo nao encontrado em {data_path}. Verifique o download e o caminho do dataset."
    )

raw_data = pd.read_csv(data_path)

raw_data.head(10)

In [ ]:
raw_data.info()

In [ ]:
raw_data.describe()

## Criação de Features

In [ ]:
featured = raw_data.copy()
featured.info()

In [ ]:
featured["Interval"] = featured["Time"].diff().fillna(0)
featured["Interval"].describe()

In [ ]:
featured.head(5)

## AED

In [ ]:
from matplotlib import pyplot as plt

classes_count = featured["Class"].value_counts()
print(classes_count)

fig = plt.figure(figsize=(5, 5))
classes_count.plot(kind="bar", color="purple")
plt.title("Distribuição das classes")
plt.xlabel("Classe")
plt.ylabel("Contagem")
plt.show()


In [ ]:
def plotCorrDisp(df,correlation, x_col, y_col):
    plt.figure(figsize=(5, 5))
    plt.scatter(df[x_col], df[y_col], alpha=0.5, c=df[y_col], cmap="viridis")
    plt.title(f"Correlação entre {x_col} e {y_col}: {correlation:.3f}")
    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.show()

In [ ]:
correlation = featured["Amount"].corr(featured["Class"])

plotCorrDisp(featured, correlation, "Amount", "Class")

In [ ]:
correlation = featured["Interval"].corr(featured["Class"])

plotCorrDisp(featured, correlation, "Interval", "Class")

In [ ]:
def plotCorrMatrix(corr_matrix):
    plt.figure(figsize=(10, 8))
    plt.imshow(corr_matrix, cmap="viridis", vmin=-1, vmax=1)
    plt.colorbar()
    plt.title("Matriz de Correlação")
    plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
    plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
    plt.show()

In [ ]:
corr_matrix = featured.corr()

plotCorrMatrix(corr_matrix)

In [ ]:
corr_with_class = featured.corr()["Class"].abs().sort_values(ascending=False)
print(corr_with_class)